6/24/26 notes

# v3 — Adding a Single Self-Attention Head

This builds directly on **v2**. Steps 1–7 below (imports, hyperparameters, data, tokenizer, train/val split, batching, loss estimation) are *unchanged* — copy them straight over.

What's new:

1. A new `Head` module implementing **scaled dot-product self-attention**: every token produces a *query*, *key*, and *value* vector. We score every position against every other position (`query @ key^T`), mask out the future with a causal mask (`tril`), softmax the scores into attention weights, then mix the `value` vectors according to those weights.
2. `BigramLanguageModel` now routes the token+position embedding through this attention head (`self.att_head`) *before* the `linear_head` — so each token's representation is now a function of *all preceding tokens*, not just itself.

## 1. Imports

In [1]:
import torch
import torch.nn as nn
from torch.nn import functional as F

## 2. Hyperparameters

In [2]:
# emb_size = 32        # embedding size for each token
# batch_size = 32      # how many independent sequences will we process in parallel?
# block_size = 8       # what is the maximum context length for predictions?
# max_iters = 10000    # maximum number of iterations for training
# eval_interval = 500  # interval for evaluating the model
# learning_rate = 1e-3 # learning rate for training
# device = 'cuda' if torch.cuda.is_available() else 'cpu'
# eval_iters = 200     # number of iterations for evaluation
# seed = 42            # seed for reproducibility

# torch.manual_seed(seed)

In [3]:
# Sabine's version of the above block that does hardware detection (that way on Mac it uses GPU if available rather than CPU default)
# Detect best available device (CUDA>MPS>CPU) and set DataLoader workers accordingly
import os

emb_size = 32        # embedding size for each token
batch_size = 32      # how many independent sequences will we process in parallel?
block_size = 8       # what is the maximum context length for predictions?
max_iters = 10000    # maximum number of iterations for training
eval_interval = 500  # interval for evaluating the model
learning_rate = 1e-3 # learning rate for training
eval_iters = 200     # number of iterations for evaluation
seed = 42            # seed for reproducibility

if torch.cuda.is_available():
    device = torch.device("cuda")
    total_cores = os.cpu_count()
    NUM_WORKERS = min(8, max(1, total_cores - 2))

elif torch.backends.mps.is_available():
    device = torch.device("mps")
    NUM_WORKERS = 0  # MPS + multiprocessing can hang

else:
    device = torch.device("cpu")
    NUM_WORKERS = 0

print(f"Using device: {device}")
torch.manual_seed(seed)

Using device: cuda


## 3. Load the dataset

In [4]:
with open('./data/harry_potter.txt', encoding='utf-8') as f:
    text = f.read()

print(f"length of dataset in characters: {len(text)}")
print(text[:500])

length of dataset in characters: 5991293
THE BOY WHO LIVED Mr and Mrs Dursley of number four Privet Drive were proud to say that they were perfectly normal thank you very much .They were the last people youd expect to be involved in anything strange or mysterious because they just didnt hold with such nonsense .Mr Dursley was the director of a firm called Grunnings which made drills .He was a big beefy man with hardly any neck although he did have a very large mustache .Mrs Dursley was thin and blonde and had nearly twice the usual amo


## 4. Tokenization: characters as tokens

In [5]:
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(''.join(chars))
print(vocab_size)

 !.0123456789?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz~‘•■□
71


In [6]:
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s] # encoder: take a string, output a list of integers
decode = lambda l: ''.join([itos[i] for i in l]) # decoder: take a list of integers, output a string

print(encode("Hello there!"))
print(decode(encode("Hello there!")))

[21, 44, 51, 51, 54, 0, 59, 47, 44, 57, 44, 1]
Hello there!


## 5. Train / validation split

In [7]:
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9*len(data)) # first 90% will be train, rest val
train_data = data[:n]
val_data = data[n:]

print(data.shape, data.dtype)
print(data[:100])

torch.Size([5991293]) torch.int64
tensor([33, 21, 18,  0, 15, 28, 38,  0, 36, 21, 28,  0, 25, 22, 35, 18, 17,  0,
        26, 57,  0, 40, 53, 43,  0, 26, 57, 58,  0, 17, 60, 57, 58, 51, 44, 64,
         0, 54, 45,  0, 53, 60, 52, 41, 44, 57,  0, 45, 54, 60, 57,  0, 29, 57,
        48, 61, 44, 59,  0, 17, 57, 48, 61, 44,  0, 62, 44, 57, 44,  0, 55, 57,
        54, 60, 43,  0, 59, 54,  0, 58, 40, 64,  0, 59, 47, 40, 59,  0, 59, 47,
        44, 64,  0, 62, 44, 57, 44,  0, 55, 44])


## 6. Batching

In [8]:
def get_batch(split):
    # generate a small batch of data of inputs x and targets y
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

In [9]:
xb, yb = get_batch('train')
print('inputs:')
print(xb.shape)
print(xb)
print('targets:')
print(yb.shape)
print(yb)

inputs:
torch.Size([32, 8])
tensor([[54, 60, 57,  0, 52, 54, 59, 47],
        [ 0, 52, 40, 64,  0, 58, 59, 48],
        [44, 58, 58, 54, 57,  0, 15, 51],
        [58, 40, 48, 43,  0, 19, 44, 44],
        [58, 59, 60, 43, 44, 53, 59, 58],
        [57, 40, 48, 59,  0, 58, 62, 60],
        [ 0, 40, 42, 42, 54, 52, 55, 40],
        [ 0, 40, 58,  0, 47, 44,  0, 43],
        [40, 50, 48, 53, 46,  0, 54, 60],
        [48, 58,  0, 48, 53,  0, 45, 40],
        [51, 40, 55,  0, 55, 60, 57, 57],
        [59,  0, 47, 48, 58,  0, 55, 54],
        [57, 44,  0, 47, 40, 43,  0, 59],
        [21, 40, 57, 57, 64,  0,  2, 27],
        [41, 51, 48, 53, 50, 48, 53, 46],
        [ 0, 41, 54, 64,  0, 47, 44, 58],
        [47, 54,  0, 62, 40, 58,  0, 58],
        [44,  0, 58, 52, 48, 57, 50, 44],
        [ 0, 59, 47, 44, 64,  0, 55, 40],
        [40, 57, 57, 64,  0, 51, 40, 60],
        [40, 51, 51, 64,  0, 21, 40, 46],
        [59,  0, 52, 44,  0, 52, 54, 58],
        [47, 40, 61, 44,  0, 43, 57, 40],
      

## 7. Estimating loss

In [10]:
@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

## 8. Self-attention — what's new

This is the core idea of the transformer. For every token we compute three vectors:

- **query**: "what am I looking for?"
- **key**: "what do I contain?"
- **value**: "what do I communicate if attended to?" (if there is a match for query and key, what will I send as an output)

We score every query against every key (`query @ key^T`) to get raw attention scores, scale by `sqrt(head_size)` to keep the softmax well-behaved, **mask out future positions** with a lower-triangular mask (`tril`) so a token can only attend to itself and earlier tokens, softmax the scores into a probability distribution (`weights`), and finally take a weighted sum of the `value` vectors.

**In class:** fill in the parts marked **NEW** below.

*recall, our vocab has 71 characters. Each character has an embedding size of 32.*

In [11]:
class Head(nn.Module):
    def __init__(self, head_size): # we are defining an attention head
        super().__init__()

        # NEW: project the embedding into key, query, and value spaces (no bias)
        self.key = nn.Linear(emb_size, head_size, bias=False)
            # wx+b is how we do a linear layer (where b=bias and captures what the wx doesn't); but for simplicity, we set bias=false
        self.query = nn.Linear(emb_size, head_size, bias=False)
        self.value = nn.Linear(emb_size, head_size, bias=False)

        # NEW: causal mask so a token can only attend to itself and earlier positions
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
            # register_buffer: saves tril as fixed module state (moves with .to(device), saved in state_dict) but not a trainable parameter
            # so we mask with a lower triangular matrix with torch.tril that operates on the block size (here its 8)

    def forward(self, x): # now we define/design the attention formula we saw in our slides
        B, T, C = x.shape # batch, block, channel dim

        # NEW: compute key and query projections -> (B,T,H)
        key = self.key(x) # take input and pass through key; (B,T,H)
        query = self.query(x) # take input and pass through query; (B,T,H)

        # NEW: attention scores -- how much should each token attend to every other token?
        # scale by sqrt(d_k) so the softmax doesn't saturate
        dot_products = query @ key.transpose(-2, -1) * C ** -0.5 # multiply key and query outputs together
            # use transpose(-2,1) so we can have (BxTxH) @ (BxHxT) = BxT*T * C ** -0.5

        # NEW: mask out attention to future positions (upper triangle) before the softmax
        dot_products = dot_products.masked_fill(self.tril[:T, :T] == 0, float('-inf')) # turn future positions to -inf so softmax turns them to 0

        # NEW: turn scores into a probability distribution over positions
        weights = F.softmax(dot_products, dim=-1) # want to normalize across the key dimension (each row of the T×T matrix sums to 1)

        # NEW: mix the value vectors according to the attention weights
        value = self.value(x) # (B, T, H)
        out = weights @ value # (B, T, T) @ (B, T, H) = (B, T, H)
        return out

## 9. The model — what's new

`BigramLanguageModel` gets one new attribute, `self.att_head`, and one new line in `forward`: after combining token + position embeddings, the result is passed through the attention head *before* the `linear_head` projects it to logits. Everything else (embeddings, loss, `generate`) is unchanged from v2.

**In class:** fill in the parts marked **NEW** below.

In [12]:
class BigramLanguageModel(nn.Module):

    def __init__(self, vocab_size, emb_size):
        super().__init__()
        # each token directly reads off the logits for the next token from a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, emb_size)
        # we will get to the position embeddings later
        self.position_embedding_table = nn.Embedding(block_size, emb_size)
        # NEW: a single self-attention head (this is a new component inside our model)
        self.att_head = Head(emb_size) # for simplicity, we will keep embedding size as head size
        self.linear_head = nn.Linear(emb_size, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape

        # idx and targets are both (B,T) tensor of integers
        tok_emb = self.token_embedding_table(idx) # (B,T,C)
        pos_emb = self.position_embedding_table(torch.arange(T, device=idx.device)) # (T,C)
        x = tok_emb + pos_emb # (B,T,C)

        # NEW: let tokens attend to each other before projecting to logits
        x = self.att_head(x)
        
        logits = self.linear_head(x)  # (B,T,vocab_size)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # crop idx to the block size
            idx_cond = idx[:, -block_size:] # (B, T)

            # get the predictions
            logits, loss = self(idx_cond)
            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx

In [13]:
BigramLanguageModel(vocab_size, emb_size)

BigramLanguageModel(
  (token_embedding_table): Embedding(71, 32)
  (position_embedding_table): Embedding(8, 32)
  (att_head): Head(
    (key): Linear(in_features=32, out_features=32, bias=False)
    (query): Linear(in_features=32, out_features=32, bias=False)
    (value): Linear(in_features=32, out_features=32, bias=False)
  )
  (linear_head): Linear(in_features=32, out_features=71, bias=True)
)

## 10. Sanity check: untrained generation

Model construction is unchanged from v2 — still `BigramLanguageModel(vocab_size, emb_size)`. The `Head(emb_size)` is created internally.

In [14]:
model = BigramLanguageModel(vocab_size, emb_size)
model = model.to(device)
# print the number of parameters in the model
print(sum(p.numel() for p in model.parameters())/1e6, 'M parameters')

context = torch.zeros((1, 1), dtype=torch.long, device=device)
print(decode(model.generate(context, max_new_tokens=200)[0].tolist()))

0.007943 M parameters
 c 6G!□B8QmW□yCO2BmS4~E■2‘Xnl~cWQZQ•pCBj3‘o7wbjI3tzGxYC‘lWDO4MXR6wJkQoNvhrB5LHvHSO0XnqfhQoOr?osOt5VFHbBDx8hdJdSL IQc0OkHWGl1mMDBEhfiMor‘a1g•gZ1lTZvCGEftqkYzNUEforGEs•l7bS□8i4G~BLhiMg E■vSv4hravtNnDghbd


*inductive bias in machine learning is bias that the architecture/model is bringing on its own; there isn't that kind of bias here (?)*

## 11. Training the model

In [15]:
# create a PyTorch optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

for iter in range(max_iters):

    # every once in a while evaluate the loss on train and val sets
    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    # sample a batch of data
    xb, yb = get_batch('train')

    # evaluate the loss
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

step 0: train loss 4.2504, val loss 4.2530
step 500: train loss 2.5078, val loss 2.5021
step 1000: train loss 2.4080, val loss 2.3963
step 1500: train loss 2.3859, val loss 2.3504
step 2000: train loss 2.3511, val loss 2.3291
step 2500: train loss 2.3442, val loss 2.3225
step 3000: train loss 2.3361, val loss 2.3101
step 3500: train loss 2.3198, val loss 2.3135
step 4000: train loss 2.3233, val loss 2.3054
step 4500: train loss 2.3164, val loss 2.2982
step 5000: train loss 2.3112, val loss 2.2898
step 5500: train loss 2.3098, val loss 2.2889
step 6000: train loss 2.2991, val loss 2.2765
step 6500: train loss 2.2954, val loss 2.2860
step 7000: train loss 2.3040, val loss 2.2862
step 7500: train loss 2.2961, val loss 2.2868
step 8000: train loss 2.2865, val loss 2.2833
step 8500: train loss 2.2955, val loss 2.2836
step 9000: train loss 2.2897, val loss 2.2822
step 9500: train loss 2.2857, val loss 2.2670
step 9999: train loss 2.2821, val loss 2.2772


*this improved by .1 from v2; (the point is that adding each of our version complexities helps the model improve)*

## 12. Generate text from the trained model

In [17]:
print('''\n##########################################
# Let's generate some Harry Potter text! #
##########################################''')
context = torch.zeros((1, 1), dtype=torch.long, device=device)
print(decode(model.generate(context, max_new_tokens=1000)[0].tolist()))


##########################################
# Let's generate some Harry Potter text! #
##########################################
 cincofo durl twn sn int ws te merilevy ?shidosase Qere this youcem it llloon diced Sed hid Pragell on ses Gimer saf ms on ebblleath to acaarme !Hre as .OGrer of lwis thosloferil ats ant hing caper yther lf foupt !Hrey .Wesaicren Tasok Dove lyof arto fsowineni frro oud wed wofes .Srt Aty hetosige he shut Rouftee nersa thed Firmofant ?es whe yes .Lof owo hon mar thit ibelaly .Wheinker drouls st to yopr thick pumps ele boond ?RoblGlyof urm thrs ha cecer ars fudt archees fourthe Chato ule docist he tly .Has fo pe ance het i friveisly oubcit ?I oupe yiddoughosink .Sohes .The walf ply and wh hato adid .Ivo .I nel dre .Rong wint Deraf sumidinto whad !sousinget rers onis aldon .Ton ve thald volne lids hed .Ohato mof outrst ftart win wlild dogm 3m ath dsing buts heye the has blunk I u ipphigh she in king adre em thancat ho lfith ant as ccunlesis fof Marrgat you se b

*we are now seeing some simple, short words show up correctly*